# 02c — Baseline Attribution Graphs (base model, **v3** TF-order flip dataset)

Runs attribution graph generation on **base Gemma-2-2B** (no fine-tuning)
over the **v3** prompt set (`prompts_triangle_v3.jsonl`, **90** binary prompts:
family × tail design with geometry + numeric families). Computes structural
statistics for each prompt and saves them to `results/statistics/stats_base_v3.json`.

The v1 pilot file (`stats_base.json`) is left untouched.

**Run as a script** to avoid kernel timeouts:
```bash
jupyter nbconvert --to script 02c_baseline_attribution.ipynb
python 02c_baseline_attribution.py
```

Saves incrementally (every prompt) so a restart resumes from the last checkpoint.

**Prerequisites:** `prompts_triangle_v3.jsonl` must exist (from v2 `true_or_false` rows, **False or True** ending).

## 0 — Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere to take effect.
# Prevents CUDA allocator fragmentation on long sequential attribution loops
# (without this, OOMs appear around prompt 12 even when VRAM is nominally free).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
        print("Loaded .env")
except ImportError:
    pass

# Persistent HF cache for RunPod
if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"RunPod cache: {hf_home}")

MY_WORK = _my_work if _root else Path(".").resolve()
print(f"MY_WORK: {MY_WORK}")

## 1 — Imports & constants

In [ ]:
import json
import time
import traceback

import torch
import numpy as np

# ── Plan constants (never change) ──────────────────────────────────────────────
MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)

PHASE = "base_v3"

# ── Paths ──────────────────────────────────────────────────────────────────────
# v3: TF-order-flip subset (90 prompts) and separate stats file.
# Pilot stats (stats_base.json from v1 run) are left untouched.
DATASET_FILE   = MY_WORK / "data" / "prompts_triangle_v3.jsonl"
GRAPHS_DIR     = MY_WORK / "results" / "graphs_base_v3"
STATS_FILE     = MY_WORK / "results" / "statistics" / "stats_base_v3.json"
# v3: single stats file (no shard).
SHARD_STATS_FILE = None

GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
STATS_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Dataset file  : {DATASET_FILE}")
print(f"Graphs dir    : {GRAPHS_DIR}")
print(f"Stats output  : {STATS_FILE}")
if SHARD_STATS_FILE is not None:
    print(f"Shard (resume): {SHARD_STATS_FILE}  (exists={SHARD_STATS_FILE.exists()})")
else:
    print("Shard (resume): None (v3 — no second-pod merge)")
print(f"CUDA available: {torch.cuda.is_available()}")

# ReplacementModel.from_pretrained expects a torch.device, not a plain string.
device_str = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_str)
print(f"Device        : {device}")

## 2 — Load full dataset (90 prompts, v3)

In [ ]:
analysis_prompts = []
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        analysis_prompts.append(json.loads(line.strip()))

binary = [p for p in analysis_prompts if p.get("task_type", "binary") == "binary"]
numeric = [p for p in analysis_prompts if p.get("task_type", "binary") == "numeric"]

print(f"Loaded {len(analysis_prompts)} prompts from full dataset")
print(f"  Binary : {len(binary)}")
print(f"    True : {sum(1 for p in binary if p['label'])}")
print(f"    False: {sum(1 for p in binary if not p['label'])}")
print(f"  Numeric: {len(numeric)}")
if numeric:
    print(f"    Targets: {sorted(set(p['label_token'] for p in numeric))}")
else:
    print("    Targets: []")

## 3 — Load model

In [ ]:
from circuit_tracer import ReplacementModel

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
)
tokenizer = model.tokenizer

# Verify token IDs match constants
id_true  = tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"TOKEN_TRUE id mismatch: {id_true} != {VOCAB_ID_TRUE}"
assert id_false == VOCAB_ID_FALSE, f"TOKEN_FALSE id mismatch: {id_false} != {VOCAB_ID_FALSE}"
print(f"Token IDs verified: True={id_true}, False={id_false}")

## 4 — First-token accuracy check (pre-attribution)

Quick sanity check: does the base model's top-1 token match the expected next token?
For binary prompts this is True/False accuracy; for numeric prompts this is exact token match.

In [ ]:
overall_correct = 0

binary_correct = 0
binary_total = 0
correct_true = 0
total_true = 0
correct_false = 0
total_false = 0

numeric_correct = 0
numeric_total = 0

# Keep per-prompt predictions in memory for downstream inspection/export.
first_token_predictions = []

progress_every = 10
acc_t0 = time.time()

with torch.inference_mode():
    for idx, entry in enumerate(analysis_prompts, start=1):
        # ReplacementModel (TransformerLens backend) does not expose .model;
        # use the no-op intervention path to get baseline logits.
        logits, _ = model.feature_intervention(entry["prompt"], [])
        last_logit = logits.squeeze(0)[-1, :]
        pred_id = int(last_logit.argmax().item())
        pred_token = tokenizer.decode([pred_id])

        task_type = entry.get("task_type", "binary")
        if task_type == "binary":
            label_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
            expected_response = TOKEN_TRUE if entry["label"] else TOKEN_FALSE
            binary_total += 1
            is_correct = (pred_id == label_id)
            if is_correct:
                binary_correct += 1
                overall_correct += 1
            if entry["label"]:
                total_true += 1
                if pred_id == VOCAB_ID_TRUE:
                    correct_true += 1
            else:
                total_false += 1
                if pred_id == VOCAB_ID_FALSE:
                    correct_false += 1
        else:
            # Robust handling: if label_token is multi-token (e.g. ' 9'),
            # fallback to stripped variant (e.g. '9') when single-token.
            raw_label = entry["label_token"]
            label_ids = tokenizer.encode(raw_label, add_special_tokens=False)
            expected_response = raw_label
            if len(label_ids) != 1:
                alt = raw_label.lstrip()
                alt_ids = tokenizer.encode(alt, add_special_tokens=False)
                if len(alt_ids) == 1:
                    label_ids = alt_ids
                    expected_response = alt
                else:
                    raise ValueError(
                        f"Numeric label token must resolve to a single token, got {raw_label!r} -> {label_ids}, alt {alt!r} -> {alt_ids}"
                    )
            label_id = int(label_ids[0])
            numeric_total += 1
            is_correct = (pred_id == label_id)
            if is_correct:
                numeric_correct += 1
                overall_correct += 1

        first_token_predictions.append({
            "prompt_id": entry["prompt_id"],
            "task_type": task_type,
            "expected_response": expected_response,
            "argmax_response": pred_token,
            "expected_token_id": label_id,
            "argmax_token_id": pred_id,
            "is_correct": is_correct,
        })

        if idx % progress_every == 0 or idx == len(analysis_prompts):
            elapsed = max(time.time() - acc_t0, 1e-9)
            speed = idx / elapsed
            eta = (len(analysis_prompts) - idx) / speed if speed > 0 else float("inf")
            print(
                f"[accuracy progress] {idx}/{len(analysis_prompts)} "
                f"({idx/len(analysis_prompts):.0%}) | "
                f"running overall={overall_correct/max(idx,1):.1%} | "
                f"{speed:.2f} prompts/s | ETA {eta:.1f}s"
            )

n = len(analysis_prompts)
print("Base model first-token accuracy on full dataset:")
print(f"  Overall : {overall_correct}/{n} = {overall_correct/n:.1%}")
if binary_total:
    print(f"  Binary  : {binary_correct}/{binary_total} = {binary_correct/binary_total:.1%}")
    print(f"    True  : {correct_true}/{total_true} = {correct_true/max(total_true,1):.1%}")
    print(f"    False : {correct_false}/{total_false} = {correct_false/max(total_false,1):.1%}")
if numeric_total:
    print(f"  Numeric : {numeric_correct}/{numeric_total} = {numeric_correct/numeric_total:.1%}")

if overall_correct / n > 0.80:
    print()
    print("NOTE: First-token accuracy >80%. Interpretability gains may appear smaller.")

print()
print(f"Stored per-prompt predictions in memory: first_token_predictions (n={len(first_token_predictions)})")
print("Preview (first 5):")
for row in first_token_predictions[:5]:
    print(
        f"  {row['prompt_id']}: expected={row['expected_response']!r} "
        f"argmax={row['argmax_response']!r} correct={row['is_correct']}"
    )

This is interesting, and may cause some trouble. The " 9" is tokenized to two, so we try to get the "9" token prediction. For the numeric, it appears that the most likely first token was always " " whitespace.

In [ ]:
import pandas as pd

_ft_csv = MY_WORK / "results" / "first_token" / "base_v3_predictions.csv"
_ft_csv.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(first_token_predictions).to_csv(_ft_csv, index=False)
print(f"First-token rows saved → {_ft_csv}")

## 4b — First-token probability tables (monitoring)

Same as **`02`** §4b on the **base** **`ReplacementModel`**: binary softmax pass; **v3 has no numeric rows**, so the digit pass is skipped. Saves **`base_v3_*`** CSVs next to predictions.

In [ ]:
import pandas as pd
import torch.nn.functional as F
from IPython.display import display

PROB_MONITOR_EVERY = max(5, max(1, len(analysis_prompts)) // 20)


def _last_vocab_probs(prompt: str) -> torch.Tensor:
    logits, _ = model.feature_intervention(prompt, [])
    last = logits.squeeze(0)[-1, :].float()
    return F.softmax(last, dim=-1)


def _resolve_digit_vocab_id(d: int) -> int:
    """Single-token vocab id Gemma assigns to digit continuation (plain or spaced)."""
    ch = str(int(d))
    for variant in (ch, f" {ch}"):
        tids = tokenizer.encode(variant, add_special_tokens=False)
        if len(tids) == 1:
            return int(tids[0])
    raise ValueError(f"Digit {d!r} must map to a single token; tried {ch!r} and {(' ' + ch)!r}")


DIGIT_IDS = [_resolve_digit_vocab_id(d) for d in range(10)]
print(f"Digit vocab ids [0..9]: {DIGIT_IDS}")

binary_list = [p for p in analysis_prompts if p.get("task_type", "binary") == "binary"]
numeric_list = [p for p in analysis_prompts if p.get("task_type", "binary") == "numeric"]

# --- Pass 1: binary ---
t0 = time.time()
binary_rows = []
for i, entry in enumerate(binary_list, 1):
    probs = _last_vocab_probs(entry["prompt"])
    pt = float(probs[VOCAB_ID_TRUE].item())
    pf = float(probs[VOCAB_ID_FALSE].item())
    binary_rows.append(
        {
            "prompt_id": entry["prompt_id"],
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry.get("label"),
            "task_type": entry.get("task_type", "binary"),
            "prob_true": pt,
            "prob_false": pf,
            "prob_mass_tf": pt + pf,
        }
    )
    if PROB_MONITOR_EVERY and (i % PROB_MONITOR_EVERY == 0 or i == len(binary_list)):
        dt = max(time.time() - t0, 1e-6)
        print(
            f"  binary {i}/{len(binary_list)}  "
            f"{i/dt:.2f}/s  ETA {(len(binary_list) - i) / max(i / dt, 1e-6):.1f}s"
        )

t1 = time.time()
df_binary_token_probs = pd.DataFrame(binary_rows)
print(f"df_binary_token_probs rows={len(df_binary_token_probs)}  wall={t1-t0:.1f}s")

# --- Pass 2: numeric digit masses (skipped for v3 binary-only) ---
if not numeric_list:
    df_numeric_digit_probs = pd.DataFrame()
    print("df_numeric_digit_probs rows=0  (no numeric prompts in v3)")
else:
    numeric_rows = []
    t0 = time.time()
    for i, entry in enumerate(numeric_list, 1):
        probs = _last_vocab_probs(entry["prompt"])
        row = {
            "prompt_id": entry["prompt_id"],
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry.get("label"),
            "label_token": entry.get("label_token"),
            "task_type": entry.get("task_type"),
        }
        mass = 0.0
        for d in range(10):
            p = float(probs[DIGIT_IDS[d]].item())
            row[f"prob_digit_{d}"] = p
            mass += p
        row["prob_mass_digits"] = mass
        numeric_rows.append(row)
        if PROB_MONITOR_EVERY and (i % PROB_MONITOR_EVERY == 0 or i == len(numeric_list)):
            dt = max(time.time() - t0, 1e-6)
            print(
                f"  numeric {i}/{len(numeric_list)}  "
                f"{i/dt:.2f}/s  ETA {(len(numeric_list) - i) / max(i / dt, 1e-6):.1f}s"
            )

    t1 = time.time()
    df_numeric_digit_probs = pd.DataFrame(numeric_rows)
    print(f"df_numeric_digit_probs rows={len(df_numeric_digit_probs)}  wall={t1-t0:.1f}s")

display(df_binary_token_probs.head(3))
if not df_numeric_digit_probs.empty:
    display(df_numeric_digit_probs.head(3))

FIRST_TOKEN_DIR = MY_WORK / "results" / "first_token"
FIRST_TOKEN_DIR.mkdir(parents=True, exist_ok=True)
stem = f"{PHASE}_v3"
path_bin = FIRST_TOKEN_DIR / f"{stem}_binary_token_probs.csv"
path_num = FIRST_TOKEN_DIR / f"{stem}_numeric_digit_probs.csv"
df_binary_token_probs.to_csv(path_bin, index=False)
if not df_numeric_digit_probs.empty:
    df_numeric_digit_probs.to_csv(path_num, index=False)
    print(f"Wrote {path_num}")
print(f"Wrote {path_bin}")

In [ ]:
pd.DataFrame(first_token_predictions)

## 5 — Attribution + stats (coupled, storage-safe)

Runs attribution for each prompt and computes stats **immediately in one pass** —
graph objects are discarded after stats are extracted (no full graph persisted to disk).

**Artifact policy (`N_FULL_GRAPH = 3` for v3):**
- First **3** prompts (dataset order): save `.pt` + frontend JSON for inspection/figures.
- All prompts: compute and append stats to `stats_base_v3.json` only.
- Remaining prompts: stats-only, no large graph files written.

**Resume:** successfully attributed `prompt_id`s in **`stats_base_v3.json`** are skipped. New rows append only to that file.

In [ ]:
import sys, importlib
sys.modules.pop("utils.graph_statistics", None)
import utils.graph_statistics as gs
importlib.reload(gs)

from circuit_tracer import attribute
from circuit_tracer.utils import create_graph_files
from utils.graph_statistics import compute_statistics, load_statistics, append_statistic

# Number of prompts for which full graph artifacts (.pt + JSON) are kept.
# All others get stats-only extraction (no large disk writes).
N_FULL_GRAPH = 3

def _resolve_attr_targets(entry: dict) -> list[str]:
    task_type = entry.get("task_type", "binary")
    if task_type == "binary":
        return [TOKEN_TRUE, TOKEN_FALSE]
    raw_label = entry["label_token"]
    ids_raw = tokenizer.encode(raw_label, add_special_tokens=False)
    if len(ids_raw) == 1:
        return [raw_label]
    alt = raw_label.lstrip()
    ids_alt = tokenizer.encode(alt, add_special_tokens=False)
    if len(ids_alt) == 1:
        return [alt]
    raise ValueError(
        f"Numeric attribution target must be single-token, "
        f"got {raw_label!r}->{ids_raw}, alt {alt!r}->{ids_alt}"
    )

def _graph_pt_path(prompt_id: str) -> Path:
    return GRAPHS_DIR / prompt_id / f"{prompt_id}.pt"

print(f"N_FULL_GRAPH     : {N_FULL_GRAPH}  (full artifacts for first {N_FULL_GRAPH} prompts)")
print(f"Module file      : {gs.__file__}")
print(f"Robust load_stat : {'if not raw.strip():' in Path(gs.__file__).read_text()}")

In [ ]:
# ── Resume: skip successfully-attributed prompts (main + optional 02b shard) ─
# Prompts with attribution_succeeded False in the MAIN file are retried here.
# Rows still append only to STATS_FILE (shard is read-only for done_ids).

def _done_prompt_ids(path) -> set[str]:
    return {
        e["prompt_id"]
        for e in load_statistics(path)
        if e.get("prompt_id") is not None and e.get("attribution_succeeded")
    }

existing_stats = load_statistics(STATS_FILE)
done_main = _done_prompt_ids(STATS_FILE)
done_shard: set[str] = set()
if SHARD_STATS_FILE is not None and SHARD_STATS_FILE.exists():
    done_shard = _done_prompt_ids(SHARD_STATS_FILE)

done_ids = done_main | done_shard
n_failed_prev = sum(1 for e in existing_stats if not e.get("attribution_succeeded"))

print(f"Succeeded in {STATS_FILE.name}     : {len(done_main)}")
if SHARD_STATS_FILE is not None and SHARD_STATS_FILE.exists():
    print(f"Succeeded in {SHARD_STATS_FILE.name}: {len(done_shard)}")
else:
    print(f"Shard (disabled)                    : {SHARD_STATS_FILE}")
print(f"Union skip set                     : {len(done_ids)}")
print(f"Previously failed (main file)    : {n_failed_prev} (retried if still in remaining)")

remaining = [p for p in analysis_prompts if p["prompt_id"] not in done_ids]
print(f"Remaining    : {len(remaining)} prompts")

# Full artifacts are saved only for the first N_FULL_GRAPH prompts (by dataset order).
# Determine which prompt IDs qualify, regardless of what is already done.
full_artifact_ids = {p["prompt_id"] for p in analysis_prompts[:N_FULL_GRAPH]}
print(f"Full-artifact set: {sorted(full_artifact_ids)}")

In [ ]:
import gc

# ── Coupled attribution + stats loop ──────────────────────────────────────────
t0 = time.time()
n_success = 0
n_fail = 0
progress_every = 5

for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    task_type = entry.get("task_type", "binary")
    save_full = pid in full_artifact_ids
    print(f"[{i}/{len(remaining)}] {pid} ({task_type})"
          f"{' [full artifact]' if save_full else ''} ...", end=" ", flush=True)

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=model,
            attribution_targets=_resolve_attr_targets(entry),
            **ATTR_KWARGS,
        )

        # Compute stats immediately — before any optional disk I/O.
        stat = compute_statistics(graph, entry, phase=PHASE)
        append_statistic(stat, STATS_FILE)
        n_success += 1

        # Full artifact save for inspection subset only.
        if save_full:
            graph_out = GRAPHS_DIR / pid
            graph_out.mkdir(parents=True, exist_ok=True)
            graph.to_pt(str(_graph_pt_path(pid)))
            create_graph_files(graph, slug=pid, output_path=str(graph_out))

        # Discard large graph tensor, then aggressively free CUDA memory to
        # prevent allocator fragmentation across 300 sequential attribution calls.
        del graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        density = stat.get("edge_density")
        t20 = stat.get("topk20") or {}
        ls = stat.get("layer_stats") or {}
        print(
            f"OK  n_active={stat.get('n_active_features')}"
            + (f"  density={density:.4f}" if density is not None else "  density=N/A")
            + (f"  topk20={t20.get('score_total'):.3f}" if t20.get('score_total') is not None else "")
            + (f"  ent={ls.get('entropy_bits'):.2f}b" if ls.get('entropy_bits') is not None else "")
            + (" [saved .pt+JSON]" if save_full else "")
        )

    except Exception as exc:
        # Failed rows are NOT added to done_ids so resume will retry them.
        stat = {
            "prompt_id": pid,
            "phase": PHASE,
            "task_type": task_type,
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry["label"],
            "label_token": entry["label_token"],
            "template_id": entry.get("template_id"),
            "attribution_succeeded": False,
            "_error": str(exc),
        }
        append_statistic(stat, STATS_FILE)
        n_fail += 1
        print(f"FAIL {exc}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if i % progress_every == 0 or i == len(remaining):
        elapsed = max(time.time() - t0, 1e-9)
        speed = i / elapsed
        eta = (len(remaining) - i) / speed if speed > 0 else float("inf")
        print(
            f"[progress] {i}/{len(remaining)} ({i/len(remaining):.0%}) | "
            f"success={n_success/max(i,1):.1%} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

print()
print(f"Done. Elapsed: {time.time() - t0:.0f}s")
print(f"  Successes : {n_success}")
print(f"  Failures  : {n_fail}")

## 6 — Summary and go/no-go check

In [ ]:
from utils.graph_statistics import load_statistics, aggregate_statistics
import json

all_stats = load_statistics(STATS_FILE)
agg = aggregate_statistics(all_stats)

print(f"Total prompts in stats file : {agg['n_total']}")
print(f"Successfully attributed     : {agg['n_succeeded']}")
print(f"Success rate                : {agg['success_rate']:.1%}")
print()

# ── Legacy scalar block ────────────────────────────────────────────────────────
legacy_metrics = [
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy', 'mean_error_node_weight',
    'logit_gap',
]
print("── Legacy metrics ─────────────────────────────────────────────────────")
print(f"{'Metric':<32} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 76)
for m in legacy_metrics:
    v = agg.get(m)
    if v:
        print(f"{m:<32} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<32} {'N/A':>10}")

print()

# ── Supervisor layer_stats block ───────────────────────────────────────────────
print("── Supervisor: layer scalar summary ───────────────────────────────────")
print(f"{'Metric':<32} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 76)
for m in ['layer_stats_mean', 'layer_stats_std', 'layer_stats_median', 'layer_stats_entropy_bits']:
    v = agg.get(m)
    if v:
        print(f"{m:<32} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<32} {'N/A':>10}")

print()

# ── Supervisor topk20 scalars ──────────────────────────────────────────────────
print("── Supervisor: top-K=20 influence scalars ──────────────────────────────")
for m in ['topk20_score_total', 'topk20_score_gini']:
    v = agg.get(m)
    if v:
        print(f"{m:<32} mean={v['mean']:.4f}  std={v['std']:.4f}  IQR={v['iqr']:.4f}")
    else:
        print(f"{m:<32} N/A")

print()

# ── Supervisor: prune_curve summary (mean node/edge counts per threshold) ──────
pc = agg.get("prune_curve_agg")
if pc:
    print("── Supervisor: prune curve (mean across prompts) ───────────────────────")
    print(f"{'Threshold':>10} {'Mean nodes':>12} {'Mean edges':>12} {'Mean density':>14}")
    print("-" * 52)
    for t_str, vals in sorted(pc.items(), key=lambda x: float(x[0])):
        mn = vals["mean_n_nodes"]
        me = vals["mean_n_edges"]
        md = vals["mean_density"]
        print(
            f"{float(t_str):>10.2f} "
            f"{mn if mn is None else f'{mn:>12.1f}'} "
            f"{me if me is None else f'{me:>12.1f}'} "
            f"{md if md is None else f'{md:>14.6f}'}"
        )
    print()

# ── Go/no-go check ─────────────────────────────────────────────────────────────
sr = agg['success_rate']
if sr < 0.70:
    print("STOP: Success rate below 70%. Investigate prompt set before proceeding.")
elif sr < 0.85:
    print(f"NOTE: Success rate {sr:.1%} is below 85%. Proceed but flag as limitation.")
else:
    print(f"Success rate {sr:.1%} — OK for structural analysis (03) on v3 stats if desired.")

## 7 — Scale-up note (from v2 pilot; not used for v3)

After the pilot: if **`n_success >= N_PILOT * 0.9`** (or whenever you want the full probe set),

set **`RUN_FULL = True`** in §1 and **re-run from §1**.

§5 skips IDs already recorded in **`stats_lora_combined.json`** (if any) **and**

**reuses pilot successes by copying rows** — no duplicate attribution cost for prompts that finished during the pilot.